In [1]:
# ============================================================
# CELL 1 — IMPORT
# ============================================================

import pandas as pd
import numpy as np
import re
import shutil
import zipfile
import html

from pathlib import Path
from datetime import datetime, date
from xml.etree import ElementTree as ET

from openpyxl import load_workbook
from openpyxl.utils.datetime import to_excel

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ============================================================
# CELL 2 — FILE PATH
# ============================================================

BASE_DIR = Path(
    r"C:\Users\zerox\OneDrive\Documents\Desktop\learning"
)

# ------------------------------------------------------------
# MASTER TEMPLATE
# ------------------------------------------------------------

MASTER_FILE = (
    BASE_DIR
    / "data"
    / "master"
    / "Learning agility self assesment & superior_24.7.2026.xlsx"
)

# ------------------------------------------------------------
# RAW ASSESSMENT
# ------------------------------------------------------------

SELF_FILE = (
    BASE_DIR
    / "data"
    / "raw"
    / "Self Assessment MASTER Questions Answers 1.xlsx"
)

SUPERIOR_FILE = (
    BASE_DIR
    / "data"
    / "raw"
    / "Superior Assessment MASTER Questions Answers.xlsx"
)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed"
    / "master_output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("========================================")
print("PATH CONFIGURATION")
print("========================================")

print("Master   :", MASTER_FILE)
print("Exists   :", MASTER_FILE.exists())

print("\nSelf     :", SELF_FILE)
print("Exists   :", SELF_FILE.exists())

print("\nSuperior :", SUPERIOR_FILE)
print("Exists   :", SUPERIOR_FILE.exists())

print("\nOutput   :", OUTPUT_DIR)
print("Exists   :", OUTPUT_DIR.exists())

PATH CONFIGURATION
Master   : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\master\Learning agility self assesment & superior_24.7.2026.xlsx
Exists   : True

Self     : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\raw\Self Assessment MASTER Questions Answers 1.xlsx
Exists   : True

Superior : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\raw\Superior Assessment MASTER Questions Answers.xlsx
Exists   : True

Output   : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\processed\master_output
Exists   : True


In [3]:
# ============================================================
# CELL 3 — LOAD MASTER + RAW
# ============================================================

assert MASTER_FILE.exists(), (
    f"Master file tidak ditemukan:\n{MASTER_FILE}"
)

assert SELF_FILE.exists(), (
    f"Self Assessment file tidak ditemukan:\n{SELF_FILE}"
)

assert SUPERIOR_FILE.exists(), (
    f"Superior Assessment file tidak ditemukan:\n{SUPERIOR_FILE}"
)

# ------------------------------------------------------------
# LOAD MASTER
# ------------------------------------------------------------

master_wb = load_workbook(
    MASTER_FILE,
    data_only=False
)

# ------------------------------------------------------------
# LOAD RAW SELF
# ------------------------------------------------------------

df_self = pd.read_excel(
    SELF_FILE,
    engine="openpyxl"
)

# ------------------------------------------------------------
# LOAD RAW SUPERIOR
# ------------------------------------------------------------

df_superior = pd.read_excel(
    SUPERIOR_FILE,
    engine="openpyxl"
)

print("========================================")
print("FILES LOADED")
print("========================================")

print("Self     :", df_self.shape)
print("Superior :", df_superior.shape)

print("\nMaster sheets:")

for sheet in master_wb.sheetnames:
    print("-", sheet)

c:\Users\zerox\OneDrive\Documents\Desktop\learning\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


FILES LOADED
Self     : (32, 110)
Superior : (30, 109)

Master sheets:
- Identitas
- mental agility
- people agility
- change agility
- result agility
- self awareness
- Profile Summary
- Kompetensi
- sheet template
- Dasboard
- Aktivitas pengembangan
- Bukti Aktivitas Pengembangan
- Penilaian Perilaku
- Form Penilaian Presentasi
- panduan


In [4]:
# ============================================================
# CELL 4 — TEXT NORMALIZATION
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value)

    # Non-breaking space
    value = value.replace("\xa0", " ")

    # Line breaks
    value = value.replace("\n", " ")
    value = value.replace("\r", " ")

    # Multiple spaces
    value = re.sub(r"\s+", " ", value)

    return value.strip().lower()

In [5]:
# ============================================================
# CELL 5 — EXTRACT MASTER 5 DIMENSIONS
# ============================================================

DIMENSIONS = {
    "mental_agility": "mental agility",
    "people_agility": "people agility",
    "change_agility": "change agility",
    "result_agility": "result agility",
    "self_awareness": "self awareness",
}

indicator_master = []

for dimension_key, sheet_name in DIMENSIONS.items():

    ws = master_wb[sheet_name]

    # Master memiliki 10 indikator:
    # Row 13 = indikator 1
    # Row 14 = indikator 2
    # ...
    # Row 22 = indikator 10

    for indicator_no, row in enumerate(range(13, 23), start=1):

        indicator = ws.cell(row, 2).value
        question = ws.cell(row, 3).value

        levels = {
            1: ws.cell(row, 4).value,
            2: ws.cell(row, 5).value,
            3: ws.cell(row, 6).value,
            4: ws.cell(row, 7).value,
            5: ws.cell(row, 8).value,
        }

        indicator_master.append({

            # ------------------------------------------------
            # IDENTITAS
            # ------------------------------------------------

            "dimension": dimension_key,

            "sheet_name": sheet_name,

            "indicator_no": indicator_no,

            "indicator_id": (
                f"{dimension_key}_{indicator_no:02d}"
            ),

            # ------------------------------------------------
            # INDICATOR
            # ------------------------------------------------

            "indicator": indicator,

            # ------------------------------------------------
            # QUESTION
            # ------------------------------------------------

            "question": question,

            "question_normalized": normalize_text(question),

            # ------------------------------------------------
            # LEVEL 1 - 5
            # ------------------------------------------------

            "level_1": levels[1],
            "level_2": levels[2],
            "level_3": levels[3],
            "level_4": levels[4],
            "level_5": levels[5],

            # ------------------------------------------------
            # NORMALIZED ANSWERS
            # ------------------------------------------------

            "level_1_normalized": normalize_text(levels[1]),
            "level_2_normalized": normalize_text(levels[2]),
            "level_3_normalized": normalize_text(levels[3]),
            "level_4_normalized": normalize_text(levels[4]),
            "level_5_normalized": normalize_text(levels[5]),
        })


# ============================================================
# CREATE DATAFRAME
# ============================================================

indicator_master_df = pd.DataFrame(indicator_master)


# ============================================================
# DISPLAY STRUCTURE
# ============================================================

print("========================================")
print("MASTER STRUCTURE")
print("========================================")

print(
    "Total indicators :",
    len(indicator_master_df)
)

print("\nIndicators per dimension:")

print(
    indicator_master_df
    .groupby("dimension")
    .size()
)

print("\nIndicator IDs:")

print(
    indicator_master_df[
        [
            "dimension",
            "indicator_no",
            "indicator_id",
            "indicator"
        ]
    ].to_string(index=False)
)

MASTER STRUCTURE
Total indicators : 50

Indicators per dimension:
dimension
change_agility    10
mental_agility    10
people_agility    10
result_agility    10
self_awareness    10
dtype: int64

Indicator IDs:
     dimension  indicator_no      indicator_id                                                                                     indicator
mental_agility             1 mental_agility_01                                            Analytical Thinking\n(memahami data & fakta dasar)
mental_agility             2 mental_agility_02                                               Root Cause Analysis\n(mencari penyebab masalah)
mental_agility             3 mental_agility_03                          Critical Thinking & Inquisitiveness\n(mempertanyakan & mengevaluasi)
mental_agility             4 mental_agility_04                            Explaining Complexity\n(menjelaskan hal kompleks secara sederhana)
mental_agility             5 mental_agility_05                                    Int

In [6]:
# ============================================================
# CELL 6 — VALIDATE MASTER
# ============================================================

assert len(indicator_master_df) == 50, (
    f"Expected 50 indicators, "
    f"found {len(indicator_master_df)}"
)

assert (
    indicator_master_df["indicator_id"]
    .nunique()
    == 50
), "Duplicate indicator_id detected."

assert (
    indicator_master_df["question_normalized"]
    .nunique()
    == 50
), "Duplicate questions detected."

for col in [
    "level_1",
    "level_2",
    "level_3",
    "level_4",
    "level_5"
]:

    missing = (
        indicator_master_df[col]
        .isna()
        .sum()
    )

    assert missing == 0, (
        f"{col} has {missing} missing values."
    )


print("========================================")
print("MASTER VALIDATION")
print("========================================")
print("50 indicators       : OK")
print("50 unique questions : OK")
print("Level 1-5 complete  : OK")

MASTER VALIDATION
50 indicators       : OK
50 unique questions : OK
Level 1-5 complete  : OK


In [7]:
# ============================================================
# CELL 7 — DETECT RAW QUESTION COLUMNS
# ============================================================

def detect_question_columns(df, master_questions):
    master_question_set = {
        normalize_text(q) for q in master_questions if pd.notna(q)
    }

    question_columns = []
    for col in df.columns:
        normalized_col = normalize_text(col)
        if normalized_col in master_question_set:
            question_columns.append(col)

    return question_columns


master_questions = indicator_master_df["question"].tolist()

self_question_cols = detect_question_columns(df_self, master_questions)
superior_question_cols = detect_question_columns(df_superior, master_questions)

print("========================================")
print("RAW QUESTION VALIDATION")
print("========================================")

print("Self questions detected     :", len(self_question_cols))
print("Superior questions detected :", len(superior_question_cols))

assert len(self_question_cols) == 50, (
    f"Self raw contains {len(self_question_cols)} questions, expected 50 Master questions."
)

assert len(superior_question_cols) == 50, (
    f"Superior raw contains {len(superior_question_cols)} questions, expected 50 Master questions."
)

print("\nQuestion structure: OK")

RAW QUESTION VALIDATION
Self questions detected     : 50
Superior questions detected : 50

Question structure: OK


In [8]:
# ============================================================
# CELL 8 — MASTER ANSWER → RATING MAPPING
# ============================================================

answer_mapping = {}

for _, row in indicator_master_df.iterrows():

    question_key = row["question_normalized"]

    answer_mapping[question_key] = {

        normalize_text(row["level_1"]): 1,

        normalize_text(row["level_2"]): 2,

        normalize_text(row["level_3"]): 3,

        normalize_text(row["level_4"]): 4,

        normalize_text(row["level_5"]): 5,
    }


print("========================================")
print("ANSWER MAPPING")
print("========================================")

print(
    "Questions mapped :",
    len(answer_mapping)
)

print(
    "Answers mapped   :",
    sum(
        len(v)
        for v in answer_mapping.values()
    )
)

assert len(answer_mapping) == 50

print("50 questions × 5 ratings = 250 mappings")

ANSWER MAPPING
Questions mapped : 50
Answers mapped   : 250
50 questions × 5 ratings = 250 mappings


In [9]:
# ============================================================
# CELL 9 — RAW → SCORED DATA FUNCTION
# ============================================================

def clean_value(value):
    """Membersihkan value dari Excel. NaN / None → string kosong."""
    if pd.isna(value):
        return ""
    return str(value).strip()


def process_assessment(df, question_columns, rater_type):
    results = []

    for row_index, row in df.iterrows():
        # Clean helper for row dict
        row_dict = {normalize_text(k): v for k, v in row.items()}

        if rater_type == "Self":
            employee_name = clean_value(row.get("Nama", row_dict.get("nama", "")))
            superior_name = ""
        else:
            employee_name = clean_value(
                row.get("Nama bawahan yang dinilai", row_dict.get("nama bawahan yang dinilai", ""))
            )
            superior_name = clean_value(row.get("Nama", row_dict.get("nama", "")))

        email = clean_value(row.get("Email", row_dict.get("email", "")))
        division = clean_value(row.get("Divisi", row_dict.get("divisi", "")))
        unit = clean_value(row.get("Unit", row_dict.get("unit", "")))
        department = clean_value(row.get("Departemen", row_dict.get("departemen", "")))
        directorate = clean_value(row.get("Direktorat", row_dict.get("direktorat", "")))
        current_level = clean_value(row.get("Level Saat Ini", row_dict.get("level saat ini", "")))
        target_level = clean_value(row.get("Level Tujuan", row_dict.get("level tujuan", "")))
        next_path = clean_value(row.get("Level Next Path", row_dict.get("level next path", "")))
        completion_time = row.get("Completion time", row_dict.get("completion time", None))
        last_modified_time = row.get("Last modified time", row_dict.get("last modified time", None))

        for question_col in question_columns:
            question_normalized = normalize_text(question_col)
            answer = row.get(question_col, "")

            if pd.isna(answer):
                continue

            answer_str = str(answer).strip()
            if not answer_str:
                continue

            answer_normalized = normalize_text(answer_str)

            master_rows = indicator_master_df[
                indicator_master_df["question_normalized"] == question_normalized
            ]

            if master_rows.empty:
                results.append({
                    "rater_type": rater_type,
                    "employee_name": employee_name,
                    "superior_name": superior_name,
                    "email": email,
                    "division": division,
                    "unit": unit,
                    "department": department,
                    "directorate": directorate,
                    "current_level": current_level,
                    "target_level": target_level,
                    "next_path": next_path,
                    "completion_time": completion_time,
                    "last_modified_time": last_modified_time,
                    "question": question_col,
                    "answer_text": answer_str,
                    "rating": np.nan,
                    "mapping_status": "QUESTION_NOT_FOUND",
                    "indicator_id": "",
                    "dimension": "",
                    "indicator_no": np.nan,
                    "indicator": "",
                })
                continue

            master_row = master_rows.iloc[0]
            rating = answer_mapping[question_normalized].get(answer_normalized, np.nan)
            mapping_status = "ANSWER_NOT_FOUND" if pd.isna(rating) else "MATCHED"

            results.append({
                "rater_type": rater_type,
                "employee_name": employee_name,
                "superior_name": superior_name,
                "email": email,
                "division": division,
                "unit": unit,
                "department": department,
                "directorate": directorate,
                "current_level": current_level,
                "target_level": target_level,
                "next_path": next_path,
                "completion_time": completion_time,
                "last_modified_time": last_modified_time,
                "question": question_col,
                "answer_text": answer_str,
                "rating": rating,
                "mapping_status": mapping_status,
                "indicator_id": master_row["indicator_id"],
                "dimension": master_row["dimension"],
                "indicator_no": master_row["indicator_no"],
                "indicator": master_row["indicator"],
            })

    return pd.DataFrame(results)

In [10]:
# ============================================================
# CELL 10 — PROCESS SELF
# ============================================================

self_scored_df = process_assessment(
    df=df_self,
    question_columns=self_question_cols,
    rater_type="Self"
)

print("========================================")
print("SELF ASSESSMENT")
print("========================================")

print(
    "Total records :",
    len(self_scored_df)
)

print(
    "\nMapping status:"
)

print(
    self_scored_df[
        "mapping_status"
    ].value_counts()
)

SELF ASSESSMENT
Total records : 1600

Mapping status:
mapping_status
MATCHED    1600
Name: count, dtype: int64


In [11]:
# ============================================================
# CELL 11 — PROCESS SUPERIOR
# ============================================================

superior_scored_df = process_assessment(
    df=df_superior,
    question_columns=superior_question_cols,
    rater_type="Superior"
)

print("========================================")
print("SUPERIOR ASSESSMENT")
print("========================================")

print(
    "Total records :",
    len(superior_scored_df)
)

print(
    "\nMapping status:"
)

print(
    superior_scored_df[
        "mapping_status"
    ].value_counts()
)

SUPERIOR ASSESSMENT
Total records : 1500

Mapping status:
mapping_status
MATCHED    1500
Name: count, dtype: int64


In [12]:
# ============================================================
# CELL 12 — COMBINE + VALIDATE SCORING
# ============================================================

scored_answers_df = pd.concat(
    [
        self_scored_df,
        superior_scored_df
    ],
    ignore_index=True
)

total_records = len(
    scored_answers_df
)

matched_records = (
    scored_answers_df[
        scored_answers_df["mapping_status"] == "MATCHED"
    ].shape[0]
)

unmatched_records = (
    total_records - matched_records
)

coverage = (
    matched_records / total_records
    if total_records > 0
    else 0
)

print("========================================")
print("SCORING VALIDATION")
print("========================================")

print("Total records :", total_records)
print("Matched       :", matched_records)
print("Unmatched     :", unmatched_records)
print("Coverage      :", f"{coverage:.2%}")

# ------------------------------------------------------------
# EMPLOYEE VALIDATION
# ------------------------------------------------------------

empty_employee = (
    scored_answers_df["employee_name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Empty employee:",
    empty_employee
)

if unmatched_records > 0:

    print(
        "\nWARNING: Ada jawaban yang tidak cocok."
    )

    display(
        scored_answers_df[
            scored_answers_df[
                "mapping_status"
            ] != "MATCHED"
        ][
            [
                "rater_type",
                "employee_name",
                "question",
                "answer_text",
                "mapping_status"
            ]
        ].head(50)
    )

else:

    print(
        "\n100% answer mapping berhasil."
    )

assert empty_employee == 0, (
    "Ada record dengan employee_name kosong."
)

SCORING VALIDATION
Total records : 3100
Matched       : 3100
Unmatched     : 0
Coverage      : 100.00%
Empty employee: 0

100% answer mapping berhasil.


In [13]:
# ============================================================
# CELL 13 — INDICATOR SCORE
# ============================================================

indicator_score_df = (
    scored_answers_df[
        scored_answers_df[
            "mapping_status"
        ] == "MATCHED"
    ]
    .groupby(
        [
            "rater_type",
            "employee_name",
            "indicator_id"
        ],
        as_index=False
    )
    ["rating"]
    .mean()
    .rename(
        columns={
            "rating": "indicator_score"
        }
    )
)

indicator_score_df = (
    indicator_score_df.merge(
        indicator_master_df[
            [
                "indicator_id",
                "dimension",
                "indicator_no",
                "indicator"
            ]
        ],
        on="indicator_id",
        how="left"
    )
)

print("========================================")
print("INDICATOR SCORE")
print("========================================")

print(
    indicator_score_df.head(20)
)

INDICATOR SCORE
   rater_type    employee_name       indicator_id  indicator_score  \
0        Self  AGUNG DWI CAHYO  change_agility_01              1.0   
1        Self  AGUNG DWI CAHYO  change_agility_02              2.0   
2        Self  AGUNG DWI CAHYO  change_agility_03              3.0   
3        Self  AGUNG DWI CAHYO  change_agility_04              4.0   
4        Self  AGUNG DWI CAHYO  change_agility_05              5.0   
5        Self  AGUNG DWI CAHYO  change_agility_06              1.0   
6        Self  AGUNG DWI CAHYO  change_agility_07              2.0   
7        Self  AGUNG DWI CAHYO  change_agility_08              3.0   
8        Self  AGUNG DWI CAHYO  change_agility_09              4.0   
9        Self  AGUNG DWI CAHYO  change_agility_10              5.0   
10       Self  AGUNG DWI CAHYO  mental_agility_01              1.0   
11       Self  AGUNG DWI CAHYO  mental_agility_02              2.0   
12       Self  AGUNG DWI CAHYO  mental_agility_03              3.0   
13  

In [14]:
# ============================================================
# CELL 14 — 5 DIMENSION SCORE
# ============================================================

dimension_score_df = (
    indicator_score_df
    .groupby(
        [
            "rater_type",
            "employee_name",
            "dimension"
        ],
        as_index=False
    )
    ["indicator_score"]
    .mean()
    .rename(
        columns={
            "indicator_score":
                "dimension_score"
        }
    )
)

print("========================================")
print("DIMENSION SCORE")
print("========================================")

display(
    dimension_score_df
)

DIMENSION SCORE


,rater_type,employee_name,dimension,dimension_score
0,Self,AGUNG DWI CAHYO,change_agility,3.0
1,Self,AGUNG DWI CAHYO,mental_agility,3.0
2,Self,AGUNG DWI CAHYO,people_agility,3.0
3,Self,AGUNG DWI CAHYO,result_agility,3.0
4,Self,AGUNG DWI CAHYO,self_awareness,3.0
...,...,...,...,...
305,Superior,Veronika Geuthredha Anastasia Noya,change_agility,3.0
306,Superior,Veronika Geuthredha Anastasia Noya,mental_agility,3.0
307,Superior,Veronika Geuthredha Anastasia Noya,people_agility,3.0
308,Superior,Veronika Geuthredha Anastasia Noya,result_agility,3.0


In [15]:
# ============================================================
# CELL 15 — SELF VS SUPERIOR GAP
# ============================================================

dimension_gap_df = (
    dimension_score_df
    .pivot_table(
        index=[
            "employee_name",
            "dimension"
        ],
        columns="rater_type",
        values="dimension_score"
    )
    .reset_index()
)

# ------------------------------------------------------------
# ENSURE COLUMNS
# ------------------------------------------------------------

if "Self" not in dimension_gap_df.columns:
    dimension_gap_df["Self"] = np.nan

if "Superior" not in dimension_gap_df.columns:
    dimension_gap_df["Superior"] = np.nan

# ------------------------------------------------------------
# GAP
# ------------------------------------------------------------

dimension_gap_df["gap"] = (
    dimension_gap_df["Self"]
    -
    dimension_gap_df["Superior"]
)

print("========================================")
print("DIMENSION GAP")
print("========================================")

display(
    dimension_gap_df
)

DIMENSION GAP


rater_type,employee_name,dimension,Self,Superior,gap
0,AGUNG DWI CAHYO,change_agility,3.0,NaN,NaN
1,AGUNG DWI CAHYO,mental_agility,3.0,NaN,NaN
2,AGUNG DWI CAHYO,people_agility,3.0,NaN,NaN
3,AGUNG DWI CAHYO,result_agility,3.0,NaN,NaN
4,AGUNG DWI CAHYO,self_awareness,3.0,NaN,NaN
...,...,...,...,...,...
195,fr,change_agility,3.0,NaN,NaN
196,fr,mental_agility,3.0,NaN,NaN
197,fr,people_agility,3.0,NaN,NaN
198,fr,result_agility,3.0,NaN,NaN


In [16]:
# CELL 16 — MASTER INPUT CELL CONFIGURATION

IDENTITY_INPUT_CELLS = [
    "D2",   # Nama
    "D3",   # Nama Atasan
    "D4",   # Level Saat Ini
    "D5",   # Level Tujuan
    "D6",   # Tanggal Assessment
    "D7",   # Unit
    "D8",   # Departemen
    "D9",   # Divisi
    "D10",  # Direktorat
    "D12",  # Level Next Path
]

# ============================================================
# ASSESSMENT INPUT
# ============================================================

ASSESSMENT_INPUT_COLUMNS = {

    # I = Self Rating
    "self_rating": 9,

    # J = Self Answer / Comment
    "self_comment": 10,

    # K = Superior Rating
    "superior_rating": 11,

    # L = Superior Answer / Comment
    "superior_comment": 12,
}


# ============================================================
# ROW INDICATOR
# ============================================================

ASSESSMENT_ROWS = list(
    range(13, 23)
)


# ============================================================
# DIMENSION SHEETS
# ============================================================

ASSESSMENT_SHEETS = list(
    DIMENSIONS.values()
)


print("========================================")
print("MASTER INPUT CONFIGURATION")
print("========================================")

print(
    "Identity input cells :",
    IDENTITY_INPUT_CELLS
)

print(
    "Assessment sheets    :",
    ASSESSMENT_SHEETS
)

print(
    "Assessment rows      :",
    f"{ASSESSMENT_ROWS[0]}-{ASSESSMENT_ROWS[-1]}"
)

print()
print("Python hanya akan mengisi input cells.")
print("Formula dan formatting mengikuti master.")

MASTER INPUT CONFIGURATION
Identity input cells : ['D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D12']
Assessment sheets    : ['mental agility', 'people agility', 'change agility', 'result agility', 'self awareness']
Assessment rows      : 13-22

Python hanya akan mengisi input cells.
Formula dan formatting mengikuti master.


In [17]:
# ============================================================
# CELL 17 — GENERATE MASTER TANPA RE-SAVE OPENPYXL
# ============================================================

# Tujuan:
# - Master Excel tetap menjadi template utama
# - Python hanya mengubah VALUE pada input cells
# - Tidak mengubah formula
# - Tidak mengubah style
# - Tidak mengubah formatting
# - Tidak mengubah merged cells
# - Tidak mengubah conditional formatting
# - Tidak mengubah data validation
# - Tidak mengubah table
# - Tidak mengubah workbook structure
#
# Cara:
# File XLSX diperlakukan sebagai ZIP package.
# Kita hanya patch XML cell yang memang boleh diisi.
# ============================================================


# ============================================================
# HELPER 1 — CLEAN EXCEL VALUE
# ============================================================

def clean_excel_value(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, str):

        value = value.strip()

        if value == "":
            return None

    return value


# ============================================================
# HELPER 2 — FIND COLUMN
# ============================================================

def find_column(df, candidates):

    if df is None or df.empty:
        return None

    normalized_columns = {
        normalize_text(col): col
        for col in df.columns
    }

    for candidate in candidates:

        candidate_normalized = normalize_text(candidate)

        if candidate_normalized in normalized_columns:

            return normalized_columns[
                candidate_normalized
            ]

    return None


# ============================================================
# HELPER 3 — REMOVE XML ATTRIBUTE
# ============================================================

def remove_xml_attribute(attrs, attribute_name):

    pattern = (
        rf'\s+{re.escape(attribute_name)}='
        rf'["\'][^"\']*["\']'
    )

    return re.sub(
        pattern,
        "",
        attrs,
        flags=re.IGNORECASE
    )


# ============================================================
# HELPER 4 — BUILD XML CELL
# ============================================================

def build_cell_xml(attrs, value):

    # Jangan mempertahankan attribute "t"
    # karena tipe data bisa berubah.
    attrs = remove_xml_attribute(
        attrs,
        "t"
    )

    # --------------------------------------------------------
    # BLANK
    # --------------------------------------------------------

    if value is None:

        return f"<c{attrs}/>"


    # --------------------------------------------------------
    # DATE / DATETIME
    # --------------------------------------------------------

    if isinstance(
        value,
        (pd.Timestamp, datetime, date)
    ):

        if isinstance(value, pd.Timestamp):

            value = value.to_pydatetime()

        serial = to_excel(value)

        return (
            f"<c{attrs}>"
            f"<v>{serial}</v>"
            f"</c>"
        )


    # --------------------------------------------------------
    # NUMBER
    # --------------------------------------------------------

    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating
        )
    ) and not isinstance(value, bool):

        if isinstance(
            value,
            (
                float,
                np.floating
            )
        ):

            number = float(value)

            if number.is_integer():

                number = int(number)

        else:

            number = int(value)

        return (
            f"<c{attrs}>"
            f"<v>{number}</v>"
            f"</c>"
        )


    # --------------------------------------------------------
    # TEXT
    # --------------------------------------------------------

    text = html.escape(
        str(value),
        quote=False
    )

    return (
        f'<c{attrs} t="inlineStr">'
        f"<is>"
        f'<t xml:space="preserve">{text}</t>'
        f"</is>"
        f"</c>"
    )


# ============================================================
# HELPER 5 — REPLACE ONE CELL DI XML
# ============================================================

def replace_cell_xml(
    xml_text,
    coordinate,
    value
):

    coordinate_pattern = re.escape(
        coordinate
    )

    pattern = re.compile(
        rf'<c'
        rf'(?P<attrs>[^>]*'
        rf'\br=["\']{coordinate_pattern}["\']'
        rf'[^>]*)>'
        rf'.*?'
        rf'</c>'
        rf'|'
        rf'<c'
        rf'(?P<attrs_self>[^>]*'
        rf'\br=["\']{coordinate_pattern}["\']'
        rf'[^>]*)'
        rf'/>',
        re.DOTALL
    )

    matches = list(
        pattern.finditer(xml_text)
    )

    if len(matches) != 1:

        raise ValueError(
            f"Cell {coordinate} "
            f"ditemukan {len(matches)} kali "
            f"di XML worksheet."
        )

    match = matches[0]

    if match.group("attrs") is not None:

        attrs = match.group("attrs")

    else:

        attrs = match.group("attrs_self")

    replacement = build_cell_xml(
        attrs,
        value
    )

    return (
        xml_text[:match.start()]
        + replacement
        + xml_text[match.end():]
    )


# ============================================================
# HELPER 6 — GET SHEET → XML PATH
# ============================================================

def get_sheet_xml_paths(xlsx_file):

    workbook_ns = (
        "http://schemas.openxmlformats.org/"
        "spreadsheetml/2006/main"
    )

    office_rel_ns = (
        "http://schemas.openxmlformats.org/"
        "officeDocument/2006/relationships"
    )

    package_rel_ns = (
        "http://schemas.openxmlformats.org/"
        "package/2006/relationships"
    )

    with zipfile.ZipFile(
        xlsx_file,
        "r"
    ) as z:

        workbook_root = ET.fromstring(
            z.read(
                "xl/workbook.xml"
            )
        )

        rels_root = ET.fromstring(
            z.read(
                "xl/_rels/workbook.xml.rels"
            )
        )


    # --------------------------------------------------------
    # RELATIONSHIP MAP
    # --------------------------------------------------------

    relationship_map = {}

    for rel in rels_root.findall(
        f"{{{package_rel_ns}}}Relationship"
    ):

        relationship_map[
            rel.attrib["Id"]
        ] = rel.attrib["Target"]


    # --------------------------------------------------------
    # SHEET MAP
    # --------------------------------------------------------

    sheet_paths = {}

    for sheet in workbook_root.findall(
        f".//{{{workbook_ns}}}sheet"
    ):

        sheet_name = sheet.attrib["name"]

        relationship_id = sheet.attrib[
            f"{{{office_rel_ns}}}id"
        ]

        target = relationship_map[
            relationship_id
        ].replace("\\", "/")


        if target.startswith("/"):

            xml_path = target.lstrip("/")

        elif target.startswith("xl/"):

            xml_path = target

        else:

            xml_path = f"xl/{target}"


        sheet_paths[
            sheet_name
        ] = xml_path


    return sheet_paths


# ============================================================
# HELPER 7 — BUILD SEMUA INPUT UNTUK 1 EMPLOYEE
# ============================================================

def build_employee_input_values(
    employee_name,
    scored_answers
):

    target_employee = normalize_text(
        employee_name
    )


    # ========================================================
    # FILTER EMPLOYEE
    # ========================================================

    employee_data = scored_answers[
        scored_answers["employee_name"]
        .apply(normalize_text)
        == target_employee
    ].copy()


    if employee_data.empty:

        raise ValueError(
            f"Tidak ada data assessment "
            f"untuk employee: {employee_name}"
        )


    # Dictionary final:
    #
    # (sheet_name, cell) -> value
    #

    values = {}


    # ========================================================
    # IDENTITAS — D2
    # ========================================================

    values[
        ("Identitas", "D2")
    ] = employee_name


    # ========================================================
    # IDENTITAS — D3 ATASAN
    # ========================================================

    superior_name = None


    subordinate_col = find_column(
        df_superior,
        [
            "Nama bawahan yang dinilai",
            "Nama Bawahan",
            "Bawahan"
        ]
    )


    superior_name_col = find_column(
        df_superior,
        [
            "Nama",
            "Name",
            "Nama atasan yang menilai",
            "Nama atasan",
            "Nama superior",
            "Superior Name"
        ]
    )


    if (
        subordinate_col is not None
        and superior_name_col is not None
    ):

        superior_rows = df_superior[
            df_superior[
                subordinate_col
            ]
            .apply(normalize_text)
            == target_employee
        ]


        if not superior_rows.empty:

            superior_name = clean_excel_value(
                superior_rows.iloc[0][
                    superior_name_col
                ]
            )


    values[
        ("Identitas", "D3")
    ] = superior_name


    # ========================================================
    # CARI DATA SELF
    # ========================================================

    self_name_col = find_column(
        df_self,
        [
            "Nama",
            "Name"
        ]
    )


    if self_name_col is None:

        raise ValueError(
            "Kolom Nama tidak ditemukan "
            "di Self Assessment."
        )


    self_rows = df_self[
        df_self[
            self_name_col
        ]
        .apply(normalize_text)
        == target_employee
    ].copy()


    if self_rows.empty:

        raise ValueError(
            f"Data Self Assessment "
            f"tidak ditemukan: {employee_name}"
        )


    self_row = self_rows.iloc[0]


    # ========================================================
    # IDENTITY MAPPING
    # ========================================================

    identity_mapping = {

        "D4": [
            "Level Saat Ini",
            "Level saat ini",
            "Current Level"
        ],

        "D5": [
            "Level Tujuan",
            "Level tujuan",
            "Target Level"
        ],

        "D6": [
            "Tanggal Assesment",
            "Tanggal Assessment",
            "Assessment Date",
            "Completion time",
            "Start time",
            "Last modified time"
        ],

        "D7": [
            "Unit"
        ],

        "D8": [
            "Departemen",
            "Department"
        ],

        "D9": [
            "Divisi",
            "Division"
        ],

        "D10": [
            "Direktorat",
            "Directorate"
        ],

        "D12": [
            "Level Next Path",
            "Next Path",
            "Level Next"
        ]
    }


    # ========================================================
    # ISI IDENTITY
    # ========================================================

    for target_cell, candidates in identity_mapping.items():

        source_col = find_column(
            self_rows,
            candidates
        )


        if source_col is None:

            values[
                ("Identitas", target_cell)
            ] = None

            continue


        values[
            ("Identitas", target_cell)
        ] = clean_excel_value(
            self_row[source_col]
        )


    # ========================================================
    # PROCESS 5 DIMENSIONS
    # ========================================================

    for dimension_key, sheet_name in DIMENSIONS.items():

        dimension_data = employee_data[
            employee_data[
                "dimension"
            ]
            == dimension_key
        ].copy()


        # ----------------------------------------------------
        # 10 INDICATORS
        # ----------------------------------------------------

        for indicator_no, row in enumerate(
            ASSESSMENT_ROWS,
            start=1
        ):

            indicator_id = (
                f"{dimension_key}_"
                f"{indicator_no:02d}"
            )


            indicator_data = dimension_data[
                dimension_data[
                    "indicator_id"
                ]
                == indicator_id
            ].copy()


            # =================================================
            # SELF
            # =================================================

            self_data = indicator_data[
                indicator_data[
                    "rater_type"
                ]
                == "Self"
            ].copy()


            self_rating = None


            if not self_data.empty:

                ratings = pd.to_numeric(
                    self_data["rating"],
                    errors="coerce"
                ).dropna()


                if not ratings.empty:

                    self_rating = ratings.iloc[0]


            self_comment = None


            if not self_data.empty:

                answers = (
                    self_data["answer_text"]
                    .dropna()
                    .astype(str)
                    .str.strip()
                )


                answers = answers[
                    answers != ""
                ]


                if not answers.empty:

                    self_comment = "\n".join(
                        answers.tolist()
                    )


            # =================================================
            # SUPERIOR
            # =================================================

            superior_data = indicator_data[
                indicator_data[
                    "rater_type"
                ]
                == "Superior"
            ].copy()


            superior_rating = None


            if not superior_data.empty:

                ratings = pd.to_numeric(
                    superior_data["rating"],
                    errors="coerce"
                ).dropna()


                if not ratings.empty:

                    superior_rating = ratings.iloc[0]


            superior_comment = None


            if not superior_data.empty:

                answers = (
                    superior_data["answer_text"]
                    .dropna()
                    .astype(str)
                    .str.strip()
                )


                answers = answers[
                    answers != ""
                ]


                if not answers.empty:

                    superior_comment = "\n".join(
                        answers.tolist()
                    )


            # =================================================
            # ONLY ALLOWED INPUT CELLS
            # =================================================

            values[
                (sheet_name, f"I{row}")
            ] = self_rating


            values[
                (sheet_name, f"J{row}")
            ] = self_comment


            values[
                (sheet_name, f"K{row}")
            ] = superior_rating


            values[
                (sheet_name, f"L{row}")
            ] = superior_comment


    return values


# ============================================================
# MAIN FUNCTION — GENERATE MASTER
# ============================================================

def generate_master_for_employee(
    employee_name,
    master_file,
    output_file,
    scored_answers
):

    # ========================================================
    # BUILD DATA INPUT
    # ========================================================

    input_values = build_employee_input_values(
        employee_name=employee_name,
        scored_answers=scored_answers
    )


    # ========================================================
    # GET SHEET XML PATH
    # ========================================================

    sheet_paths = get_sheet_xml_paths(
        master_file
    )


    required_sheets = {
        "Identitas",
        *ASSESSMENT_SHEETS
    }


    missing_sheets = (
        required_sheets
        - set(sheet_paths.keys())
    )


    if missing_sheets:

        raise ValueError(
            "Sheet target tidak ditemukan: "
            f"{sorted(missing_sheets)}"
        )


    # ========================================================
    # OUTPUT PATH
    # ========================================================

    output_file = Path(
        output_file
    )

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    temp_file = output_file.with_suffix(
        ".tmp.xlsx"
    )


    try:

        # ====================================================
        # OPEN MASTER ZIP
        # ====================================================

        with zipfile.ZipFile(
            master_file,
            "r"
        ) as zin:

            with zipfile.ZipFile(
                temp_file,
                "w"
            ) as zout:


                # --------------------------------------------
                # GROUP INPUT BY XML SHEET
                # --------------------------------------------

                grouped_inputs = {}


                for (
                    sheet_name,
                    coordinate
                ), value in input_values.items():

                    xml_path = sheet_paths[
                        sheet_name
                    ]


                    grouped_inputs.setdefault(
                        xml_path,
                        []
                    ).append(
                        (
                            coordinate,
                            value
                        )
                    )


                # --------------------------------------------
                # COPY ALL FILES
                # --------------------------------------------

                for item in zin.infolist():

                    data = zin.read(
                        item.filename
                    )


                    # ----------------------------------------
                    # PATCH ONLY TARGET SHEETS
                    # ----------------------------------------

                    if (
                        item.filename
                        in grouped_inputs
                    ):

                        xml_text = data.decode(
                            "utf-8"
                        )


                        for (
                            coordinate,
                            value
                        ) in grouped_inputs[
                            item.filename
                        ]:

                            xml_text = replace_cell_xml(
                                xml_text,
                                coordinate,
                                clean_excel_value(
                                    value
                                )
                            )


                        data = xml_text.encode(
                            "utf-8"
                        )


                    # ----------------------------------------
                    # WRITE BACK
                    # ----------------------------------------

                    zout.writestr(
                        item,
                        data
                    )


        # ====================================================
        # REPLACE OUTPUT
        # ====================================================

        if output_file.exists():

            output_file.unlink()


        shutil.move(
            str(temp_file),
            str(output_file)
        )


    finally:

        if temp_file.exists():

            temp_file.unlink()


    return output_file

In [18]:
# ============================================================
# CELL 18 — PRE-GENERATION VALIDATION
# ============================================================

print("=" * 60)
print("PRE-GENERATION VALIDATION")
print("=" * 60)


# ============================================================
# VALIDATE DIMENSIONS
# ============================================================

assert len(DIMENSIONS) == 5, (
    "Jumlah dimension harus 5."
)


# ============================================================
# VALIDATE INDICATORS
# ============================================================

assert len(ASSESSMENT_ROWS) == 10, (
    "Setiap dimension harus memiliki 10 indicator."
)


# ============================================================
# VALIDATE SCORED DATA
# ============================================================

required_columns = [
    "rater_type",
    "employee_name",
    "indicator_id",
    "dimension",
    "rating",
    "answer_text",
]

missing_columns = [
    col
    for col in required_columns
    if col not in scored_answers_df.columns
]

assert not missing_columns, (
    f"Kolom scored_answers_df missing: "
    f"{missing_columns}"
)


# ============================================================
# VALIDATE EMPLOYEE
# ============================================================

employees = sorted(
    scored_answers_df[
        scored_answers_df["rater_type"] == "Self"
    ]["employee_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != ""]
    .unique()
)


assert len(employees) > 0, (
    "Tidak ada employee Self Assessment."
)


print("Dimensions :", len(DIMENSIONS))
print("Indicators :", len(ASSESSMENT_ROWS))
print("Employees  :", len(employees))

print()
print("Pre-generation validation: OK")

PRE-GENERATION VALIDATION
Dimensions : 5
Indicators : 10
Employees  : 32

Pre-generation validation: OK


In [19]:
# ============================================================
# CELL 19 — MASTER STRUCTURE SNAPSHOT
# ============================================================

def get_master_structure(master_file):

    wb = load_workbook(
        master_file,
        data_only=False
    )

    structure = {}

    for sheet_name in wb.sheetnames:

        ws = wb[sheet_name]

        structure[sheet_name] = {

            "max_row": ws.max_row,

            "max_column": ws.max_column,

            "merged_ranges": tuple(
                sorted(
                    str(rng)
                    for rng in ws.merged_cells.ranges
                )
            ),

            "column_dimensions": {
                key: (
                    dim.width,
                    dim.hidden
                )
                for key, dim
                in ws.column_dimensions.items()
            },

            "row_dimensions": {
                key: (
                    dim.height,
                    dim.hidden
                )
                for key, dim
                in ws.row_dimensions.items()
            },
        }

    return structure


MASTER_STRUCTURE = get_master_structure(
    MASTER_FILE
)


print("=" * 60)
print("MASTER STRUCTURE SNAPSHOT")
print("=" * 60)

for sheet_name, info in MASTER_STRUCTURE.items():

    print(
        f"{sheet_name:35} "
        f"{info['max_row']} rows × "
        f"{info['max_column']} cols"
    )

print()
print("Master structure snapshot created.")

c:\Users\zerox\OneDrive\Documents\Desktop\learning\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


MASTER STRUCTURE SNAPSHOT
Identitas                           12 rows × 4 cols
mental agility                      23 rows × 16 cols
people agility                      23 rows × 16 cols
change agility                      23 rows × 16 cols
result agility                      23 rows × 16 cols
self awareness                      23 rows × 16 cols
Profile Summary                     113 rows × 9 cols
Kompetensi                          57 rows × 45 cols
sheet template                      1009 rows × 13 cols
Dasboard                            42 rows × 24 cols
Aktivitas pengembangan              21 rows × 23 cols
Bukti Aktivitas Pengembangan        33 rows × 26 cols
Penilaian Perilaku                  33 rows × 26 cols
Form Penilaian Presentasi           18 rows × 26 cols
panduan                             225 rows × 9 cols

Master structure snapshot created.


In [20]:
# ============================================================
# CELL 20 — VALIDASI STRUKTUR MASTER
# ============================================================

def get_master_structure(xlsx_file):
    """
    Mengambil struktur dasar workbook tanpa menyimpan ulang.
    """

    wb = load_workbook(
        xlsx_file,
        data_only=False
    )

    structure = {
        "sheetnames": list(wb.sheetnames),
        "sheets": {}
    }

    for ws in wb.worksheets:

        structure["sheets"][ws.title] = {
            "max_row": ws.max_row,
            "max_column": ws.max_column,
            "merged_ranges": sorted(
                str(rng)
                for rng in ws.merged_cells.ranges
            ),
            "freeze_panes": (
                str(ws.freeze_panes)
                if ws.freeze_panes is not None
                else None
            ),
        }

    return structure


def validate_output_structure(
    master_file,
    output_file
):
    """
    Membandingkan struktur dasar Master dengan output.

    Tidak membandingkan VALUE karena memang
    value input cells sengaja berubah.
    """

    master_structure = get_master_structure(
        master_file
    )

    output_structure = get_master_structure(
        output_file
    )

    errors = []

    # --------------------------------------------------------
    # 1. Sheet names
    # --------------------------------------------------------

    if (
        master_structure["sheetnames"]
        != output_structure["sheetnames"]
    ):
        errors.append(
            "Sheet names berubah."
        )

    # --------------------------------------------------------
    # 2. Per-sheet structure
    # --------------------------------------------------------

    master_sheets = master_structure[
        "sheets"
    ]

    output_sheets = output_structure[
        "sheets"
    ]

    for sheet_name in master_structure[
        "sheetnames"
    ]:

        if sheet_name not in output_sheets:
            errors.append(
                f"Sheet hilang: {sheet_name}"
            )
            continue

        master_info = master_sheets[
            sheet_name
        ]

        output_info = output_sheets[
            sheet_name
        ]

        if (
            master_info["max_row"]
            != output_info["max_row"]
        ):
            errors.append(
                f"{sheet_name}: max_row berubah "
                f"({master_info['max_row']} -> "
                f"{output_info['max_row']})"
            )

        if (
            master_info["max_column"]
            != output_info["max_column"]
        ):
            errors.append(
                f"{sheet_name}: max_column berubah "
                f"({master_info['max_column']} -> "
                f"{output_info['max_column']})"
            )

        if (
            master_info["merged_ranges"]
            != output_info["merged_ranges"]
        ):
            errors.append(
                f"{sheet_name}: merged cells berubah."
            )

        if (
            master_info["freeze_panes"]
            != output_info["freeze_panes"]
        ):
            errors.append(
                f"{sheet_name}: freeze panes berubah."
            )

    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    if errors:

        return {
            "status": "FAILED",
            "reason": (
                "Struktur workbook berubah."
            ),
            "errors": errors
        }

    return {
        "status": "PASSED",
        "reason": (
            "Struktur workbook tetap sama."
        ),
        "errors": []
    }


print("=" * 60)
print("OUTPUT STRUCTURE VALIDATION READY")
print("=" * 60)

OUTPUT STRUCTURE VALIDATION READY


In [21]:
# ============================================================
# CELL 21 — FORMULA PRESERVATION CHECK
# ============================================================

def get_formula_snapshot(file_path):

    wb = load_workbook(
        file_path,
        data_only=False
    )

    formulas = {}

    for sheet_name in wb.sheetnames:

        ws = wb[sheet_name]

        sheet_formulas = {}

        for row in ws.iter_rows():

            for cell in row:

                if (
                    isinstance(cell.value, str)
                    and cell.value.startswith("=")
                ):

                    sheet_formulas[
                        cell.coordinate
                    ] = cell.value


        formulas[sheet_name] = sheet_formulas

    return formulas


MASTER_FORMULAS = get_formula_snapshot(
    MASTER_FILE
)


def validate_formula_preservation(
    output_file,
    master_formulas
):

    output_formulas = get_formula_snapshot(
        output_file
    )


    # ========================================================
    # SHEET CHECK
    # ========================================================

    if set(output_formulas.keys()) != set(
        master_formulas.keys()
    ):

        return {
            "status": "FAILED",
            "reason": "Formula sheet structure changed."
        }


    # ========================================================
    # CHECK EACH FORMULA
    # ========================================================

    for sheet_name in master_formulas:

        master_sheet = master_formulas[
            sheet_name
        ]

        output_sheet = output_formulas[
            sheet_name
        ]


        for coordinate, master_formula in master_sheet.items():

            output_formula = output_sheet.get(
                coordinate
            )

            if output_formula != master_formula:

                return {
                    "status": "FAILED",
                    "reason": (
                        f"{sheet_name}!{coordinate} "
                        f"formula changed.\n"
                        f"Master  : {master_formula}\n"
                        f"Output  : {output_formula}"
                    )
                }


    return {
        "status": "PASSED",
        "reason": "All master formulas preserved."
    }

In [22]:
# ============================================================
# CELL 22 — OUTPUT INPUT VALIDATION
# ============================================================

def validate_employee_output(
    output_file,
    employee_name
):

    wb = load_workbook(
        output_file,
        data_only=False
    )

    ws_identity = wb["Identitas"]

    errors = []


    # ========================================================
    # CHECK NAME
    # ========================================================

    output_name = ws_identity["D2"].value

    if normalize_text(output_name) != normalize_text(
        employee_name
    ):

        errors.append(
            "Identitas!D2 tidak sesuai employee."
        )


    # ========================================================
    # CHECK 5 DIMENSIONS
    # ========================================================

    for sheet_name in ASSESSMENT_SHEETS:

        if sheet_name not in wb.sheetnames:

            errors.append(
                f"Sheet {sheet_name} tidak ditemukan."
            )


    if errors:

        return {
            "status": "FAILED",
            "errors": errors
        }


    return {
        "status": "PASSED",
        "errors": []
    }

In [23]:
# ============================================================
# CELL 24 — VALIDASI PACKAGE INTEGRITY
# ============================================================

def validate_package_integrity(
    master_file,
    output_file
):
    """
    Memastikan seluruh bagian XLSX yang TIDAK seharusnya
    diubah tetap identik dengan Master.

    Worksheet target memang boleh berubah karena berisi
    input yang diisi Python.
    """

    # --------------------------------------------------------
    # Ambil lokasi XML untuk sheet yang memang boleh berubah
    # --------------------------------------------------------

    sheet_paths = get_sheet_xml_paths(
        master_file
    )

    target_sheet_names = (
        ["Identitas"] + ASSESSMENT_SHEETS
    )

    target_sheet_xml = set()

    for sheet_name in target_sheet_names:

        if sheet_name not in sheet_paths:

            raise ValueError(
                f"Sheet '{sheet_name}' "
                f"tidak ditemukan di Master."
            )

        target_sheet_xml.add(
            sheet_paths[sheet_name]
        )

    # --------------------------------------------------------
    # Buka Master dan Output sebagai ZIP
    # --------------------------------------------------------

    with zipfile.ZipFile(
        master_file,
        "r"
    ) as master_zip:

        with zipfile.ZipFile(
            output_file,
            "r"
        ) as output_zip:

            # ------------------------------------------------
            # 1. Bandingkan daftar package files
            # ------------------------------------------------

            master_names = set(
                master_zip.namelist()
            )

            output_names = set(
                output_zip.namelist()
            )

            if master_names != output_names:

                missing_files = (
                    master_names
                    - output_names
                )

                added_files = (
                    output_names
                    - master_names
                )

                return {
                    "status": "FAILED",
                    "reason": (
                        "Isi package XLSX berubah."
                    ),
                    "errors": [
                        f"Missing files: {sorted(missing_files)}",
                        f"Added files: {sorted(added_files)}"
                    ]
                }

            # ------------------------------------------------
            # 2. Bandingkan seluruh file package
            # ------------------------------------------------
            #
            # Worksheet input BOLEH berbeda.
            # File lainnya HARUS identik.
            # ------------------------------------------------

            changed_files = []

            for filename in sorted(
                master_names
            ):

                # Sheet XML yang memang kita patch
                if filename in target_sheet_xml:
                    continue

                master_data = master_zip.read(
                    filename
                )

                output_data = output_zip.read(
                    filename
                )

                if master_data != output_data:

                    changed_files.append(
                        filename
                    )

            # ------------------------------------------------
            # 3. Result
            # ------------------------------------------------

            if changed_files:

                return {
                    "status": "FAILED",
                    "reason": (
                        "Ada package file "
                        "yang berubah di luar "
                        "worksheet input."
                    ),
                    "errors": changed_files
                }

    return {
        "status": "PASSED",
        "reason": (
            "Package XLSX tetap identik "
            "di luar worksheet input."
        ),
        "errors": []
    }


print("=" * 60)
print("PACKAGE INTEGRITY VALIDATION READY")
print("=" * 60)

PACKAGE INTEGRITY VALIDATION READY


In [24]:
# ============================================================
# CELL 25 — GENERATE MASTER UNTUK SEMUA KARYAWAN
# ============================================================

print("=" * 60)
print("MASTER GENERATION — SEMUA KARYAWAN")
print("=" * 60)


# ============================================================
# PREPARE OUTPUT
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


generated_files = []
failed_files = []


# ============================================================
# GET EMPLOYEE LIST
# ============================================================

employees = sorted(
    scored_answers_df[
        scored_answers_df["rater_type"] == "Self"
    ]["employee_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != ""]
    .unique()
)


print()
print(
    "Total employee:",
    len(employees)
)

print()


# ============================================================
# GENERATE ONE BY ONE
# ============================================================

for index, employee_name in enumerate(
    employees,
    start=1
):

    print(
        f"[{index}/{len(employees)}] "
        f"Processing: {employee_name}"
    )


    # ========================================================
    # SAFE FILE NAME
    # ========================================================

    safe_name = re.sub(
        r'[\\/*?:"<>|]',
        "_",
        employee_name
    ).strip()


    output_file = (
        OUTPUT_DIR
        / f"Learning_Agility_{safe_name}.xlsx"
    )


    try:

        # ====================================================
        # GENERATE DARI MASTER
        # ====================================================
        #
        # PENTING:
        # Tidak menggunakan:
        #
        # wb = load_workbook(...)
        # wb.save(...)
        #
        # File master hanya di-patch pada input cells.
        # ====================================================

        generate_master_for_employee(
            employee_name=employee_name,
            master_file=MASTER_FILE,
            output_file=output_file,
            scored_answers=scored_answers_df
        )


        # ====================================================
        # STRUCTURE VALIDATION
        # ====================================================

        structure_result = (
            validate_output_structure(
                output_file,
                MASTER_STRUCTURE
            )
        )


        if (
            structure_result["status"]
            != "PASSED"
        ):

            raise ValueError(
                "Structure validation failed: "
                + structure_result["reason"]
            )


        # ====================================================
        # FORMULA VALIDATION
        # ====================================================

        formula_result = (
            validate_formula_preservation(
                output_file,
                MASTER_FORMULAS
            )
        )


        if (
            formula_result["status"]
            != "PASSED"
        ):

            raise ValueError(
                "Formula validation failed: "
                + formula_result["reason"]
            )


        # ====================================================
        # PACKAGE INTEGRITY
        # ====================================================

        package_result = (
            validate_package_integrity(
                MASTER_FILE,
                output_file
            )
        )


        if (
            package_result["status"]
            != "PASSED"
        ):

            raise ValueError(
                "Package integrity validation failed: "
                + package_result["reason"]
            )


        # ====================================================
        # EMPLOYEE INPUT VALIDATION
        # ====================================================

        employee_result = (
            validate_employee_output(
                output_file,
                employee_name
            )
        )


        if (
            employee_result["status"]
            != "PASSED"
        ):

            raise ValueError(
                "Employee validation failed: "
                + str(
                    employee_result["errors"]
                )
            )


        # ====================================================
        # SUCCESS
        # ====================================================

        generated_files.append(
            output_file
        )


        print(
            "    ✓ Success"
        )

        print(
            "    ✓ Structure preserved"
        )

        print(
            "    ✓ Formula preserved"
        )

        print(
            "    ✓ Package integrity preserved"
        )

        print(
            "    ✓ Employee data validated"
        )


    except Exception as e:

        failed_files.append(
            {
                "employee": employee_name,
                "error": repr(e)
            }
        )


        print(
            "    ✗ ERROR:",
            repr(e)
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 60)
print("GENERATION SELESAI")
print("=" * 60)


print(
    "Berhasil :",
    len(generated_files),
    "/",
    len(employees)
)


print(
    "Gagal    :",
    len(failed_files)
)


print(
    "Output   :",
    OUTPUT_DIR
)


# ============================================================
# FAILED REPORT
# ============================================================

if failed_files:

    print()
    print("FAILED EMPLOYEES:")

    for item in failed_files:

        print(
            "-",
            item["employee"],
            ":",
            item["error"]
        )

MASTER GENERATION — SEMUA KARYAWAN

Total employee: 32

[1/32] Processing: AGUNG DWI CAHYO
    ✗ ERROR: ValueError('Cell D2 ditemukan 0 kali di XML worksheet.')
[2/32] Processing: Ade Rizqie Fonna
    ✗ ERROR: ValueError('Cell D2 ditemukan 0 kali di XML worksheet.')
[3/32] Processing: Agung Sedayu
    ✗ ERROR: ValueError('Cell D2 ditemukan 0 kali di XML worksheet.')
[4/32] Processing: Ana Kholifah
    ✗ ERROR: ValueError('Cell D2 ditemukan 0 kali di XML worksheet.')
[5/32] Processing: Anggit Catur Nugroho
    ✗ ERROR: ValueError('Cell D2 ditemukan 0 kali di XML worksheet.')
[6/32] Processing: Charli Sanjuan Siahaan
    ✗ ERROR: ValueError('Cell D2 ditemukan 0 kali di XML worksheet.')
[7/32] Processing: Deddie Yunawan Wicaksono Drajat
    ✗ ERROR: ValueError('Cell D2 ditemukan 0 kali di XML worksheet.')
[8/32] Processing: Dedek Saputra
    ✗ ERROR: ValueError('Cell D2 ditemukan 0 kali di XML worksheet.')
[9/32] Processing: Diah Puspita
    ✗ ERROR: ValueError('Cell D2 ditemukan 0 kali d